This notebook computes semantic similarity between TensorFlow and PyTorch API documentation using BERT embeddings.

**Workflow:**
1. Load API documentation from specified paths
2. Preprocess text data
3. Generate BERT embeddings
4. Compute cross-framework similarity scores
5. Identify and display most similar API pairs

## 1. Install and Import Dependencies

In [ ]:
# Install and Import Dependencies
%!pip install transformers torch scipy numpy pandas

import os
import numpy as np
import pandas as pd
from scipy.spatial.distance import cosine
from transformers import AutoTokenizer, AutoModel
import torch

# Set seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

## 2. Data Loading and Preprocessing

In [ ]:
def load_docs(directory, framework_name):
    """Load API documentation from directory structure
    
    Args:
        directory (str): Path to documentation root
        framework_name (str): Framework identifier
        
    Returns:
        List[dict]: API documents with metadata
    """
    docs = []
    for root, _, files in os.walk(directory):
        for file in files:
            if file.endswith('.md'):
                doc_path = os.path.join(root, file)
                with open(doc_path, 'r', encoding='utf-8') as f:
                    content = f.read()
                docs.append({
                    'framework': framework_name,
                    'api_name': os.path.splitext(file)[0],
                    'content': content,
                    'file_path': doc_path
                })
    return docs

# Load documentation
tf_docs = load_docs('data/docs-master', 'TensorFlow')
torch_docs = load_docs('data/pytorch-main/docs', 'PyTorch')

print(f"Loaded {len(tf_docs)} TensorFlow APIs")
print(f"Loaded {len(torch_docs)} PyTorch APIs")

## 3. Text Preprocessing

In [ ]:
def preprocess_text(text):
    """Clean and normalize documentation text
    
    Args:
        text (str): Raw documentation content
        
    Returns:
        str: Processed text ready for BERT
    """
    # Remove code blocks
    text = text.replace('```', '')
    # Collapse whitespace
    text = ' '.join(text.split())
    # Truncate to first 2000 characters to maintain context
    return text[:2000]

# Preprocess all documents
for doc in tf_docs + torch_docs:
    doc['processed_text'] = preprocess_text(doc['content'])

## 4. BERT Embedding Generation

In [ ]:
# Initialize BERT model
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
model = AutoModel.from_pretrained('bert-base-uncased')

def get_bert_embeddings(texts, batch_size=16):
    """Generate BERT embeddings in batch mode
    
    Args:
        texts (List[str]): List of processed texts
        batch_size (int): Number of texts per batch
        
    Returns:
        np.ndarray: Matrix of document embeddings
    """
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(
            batch, 
            padding=True, 
            truncation=True, 
            max_length=512, 
            return_tensors="pt"
        )
        
        with torch.no_grad():
            outputs = model(**inputs)
            
        # Use mean pooling of last hidden states
        batch_embeddings = outputs.last_hidden_state.mean(dim=1).numpy()
        embeddings.append(batch_embeddings)
    
    return np.vstack(embeddings)

# Generate embeddings for both frameworks
tf_embeddings = get_bert_embeddings([doc['processed_text'] for doc in tf_docs])
torch_embeddings = get_bert_embeddings([doc['processed_text'] for doc in torch_docs])

print(f"TensorFlow embedding matrix shape: {tf_embeddings.shape}")
print(f"PyTorch embedding matrix shape: {torch_embeddings.shape}")

## 5. Cross-Framework Similarity Analysis

In [ ]:
def compute_cross_similarity(matrix_a, matrix_b):
    """Compute pairwise cosine similarity between two embedding matrices
    
    Args:
        matrix_a (np.ndarray): N x D embedding matrix
        matrix_b (np.ndarray): M x D embedding matrix
        
    Returns:
        np.ndarray: N x M similarity matrix
    """
    # Normalize embeddings
    matrix_a_norm = matrix_a / np.linalg.norm(matrix_a, axis=1, keepdims=True)
    matrix_b_norm = matrix_b / np.linalg.norm(matrix_b, axis=1, keepdims=True)
    
    return np.dot(matrix_a_norm, matrix_b_norm.T)

# Compute similarity matrix
similarity_matrix = compute_cross_similarity(tf_embeddings, torch_embeddings)
print(f"Similarity matrix shape: {similarity_matrix.shape}")

## 6. Identify Top Similar API Pairs

In [ ]:
def get_top_matches(sim_matrix, tf_docs, torch_docs, top_k=10):
    """Identify top matching API pairs
    
    Args:
        sim_matrix (np.ndarray): Similarity matrix
        tf_docs (List[dict]): TensorFlow API docs
        torch_docs (List[dict]): PyTorch API docs
        top_k (int): Number of top matches to return
        
    Returns:
        pd.DataFrame: Sorted matches with metadata
    """
    matches = []
    rows, cols = sim_matrix.shape
    
    for i in range(rows):
        for j in range(cols):
            matches.append({
                'tf_api': tf_docs[i]['api_name'],
                'pytorch_api': torch_docs[j]['api_name'],
                'similarity': sim_matrix[i,j],
                'tf_path': tf_docs[i]['file_path'],
                'torch_path': torch_docs[j]['file_path']
            })
    
    df = pd.DataFrame(matches)
    return df.sort_values('similarity', ascending=False).head(top_k)

# Get and display top matches
top_matches = get_top_matches(similarity_matrix, tf_docs, torch_docs)
top_matches[['tf_api', 'pytorch_api', 'similarity']]

## 7. Detailed Match Analysis

In [ ]:
def show_match_details(row):
    """Display detailed comparison for a specific match"""
    print(f"TensorFlow API: {row['tf_api']}")
    print(f"PyTorch API: {row['pytorch_api']}")
    print(f"Similarity Score: {row['similarity']:.4f}\n")
    
    print("TensorFlow Documentation Excerpt:")
    print(tf_docs[[d['api_name'] == row['tf_api'] for d in tf_docs][0]['processed_text'][:500] + "...\n")
    
    print("PyTorch Documentation Excerpt:")
    print(torch_docs[[d['api_name'] == row['pytorch_api'] for d in torch_docs][0]['processed_text'][:500] + "...\n")

# (Debug) Inspect top match
show_match_details(top_matches.iloc[0])